In [4]:
import torch
print(torch.__version__)

2.10.0+cpu


In [6]:
from google.colab import files
uploaded = files.upload()

Saving daphnet+freezing+of+gait.zip to daphnet+freezing+of+gait.zip


In [8]:
import zipfile

with zipfile.ZipFile("daphnet+freezing+of+gait.zip", 'r') as zip_ref:
    zip_ref.extractall()

In [37]:
import os
os.listdir()

['.config',
 'daphnet+freezing+of+gait.zip',
 'dataset_fog_release',
 'sample_data']

In [40]:
import glob

files = glob.glob("dataset_fog_release/*.txt")
print(len(files))

0


In [41]:
import os
os.listdir("dataset_fog_release")

['scripts', 'README', 'dataset', 'doc']

In [42]:
import glob

files = glob.glob("dataset_fog_release/dataset/*.txt")
print(len(files))

17


In [43]:
import pandas as pd

data = []

for f in files:
    print("Loading:", f)
    df = pd.read_csv(f, sep=" ", header=None, engine="c")
    df = df[df.iloc[:, -1] != -1]
    data.append(df)

df = pd.concat(data, ignore_index=True)

print("Shape:", df.shape)

Loading: dataset_fog_release/dataset/S09R01.txt
Loading: dataset_fog_release/dataset/S08R01.txt
Loading: dataset_fog_release/dataset/S03R01.txt
Loading: dataset_fog_release/dataset/S01R02.txt
Loading: dataset_fog_release/dataset/S03R02.txt
Loading: dataset_fog_release/dataset/S10R01.txt
Loading: dataset_fog_release/dataset/S03R03.txt
Loading: dataset_fog_release/dataset/S05R01.txt
Loading: dataset_fog_release/dataset/S01R01.txt
Loading: dataset_fog_release/dataset/S07R01.txt
Loading: dataset_fog_release/dataset/S07R02.txt
Loading: dataset_fog_release/dataset/S04R01.txt
Loading: dataset_fog_release/dataset/S02R02.txt
Loading: dataset_fog_release/dataset/S06R01.txt
Loading: dataset_fog_release/dataset/S02R01.txt
Loading: dataset_fog_release/dataset/S05R02.txt
Loading: dataset_fog_release/dataset/S06R02.txt
Shape: (1917887, 11)


In [44]:
import numpy as np

X = df.iloc[:, 1:-1].values   # 9 sensor columns
y = df.iloc[:, -1].values     # labels

# convert to binary
y = (y > 0).astype(int)

print(np.unique(y))  # should be [0 1]

[0 1]


In [45]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)

In [46]:
def create_windows(X, y, window=128, stride=64):
    Xs, ys = [], []

    for i in range(0, len(X)-window, stride):
        Xs.append(X[i:i+window])
        ys.append(max(y[i:i+window]))  # if any FOG → 1

    return np.array(Xs), np.array(ys)

X, y = create_windows(X, y)

# reshape for CNN
X = np.transpose(X, (0, 2, 1))

print(X.shape, y.shape)


# 🔥 ADD THIS BLOCK RIGHT HERE
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
import torch

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True
)

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.float32),
                  torch.tensor(y_train, dtype=torch.long)),
    batch_size=32, shuffle=True
)

test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test, dtype=torch.float32),
                  torch.tensor(y_test, dtype=torch.long)),
    batch_size=32, shuffle=False
)

(29965, 9, 128) (29965,)


In [47]:
import torch
from torch.utils.data import TensorDataset, DataLoader



In [48]:
import torch.nn as nn

class FOGModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(9, 32, 5),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        self.lstm = nn.LSTM(input_size=64, hidden_size=64, batch_first=True)
        self.fc = nn.Linear(64, 2)

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h, _) = self.lstm(x)
        return self.fc(h[-1])

model = FOGModel()

In [49]:
import torch

weights = torch.tensor([1.0, 1.5])  # penalize missing FOG more
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        out = model(xb)
        loss = criterion(out, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Avg Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Avg Loss: 0.3137
Epoch 2, Avg Loss: 0.2606
Epoch 3, Avg Loss: 0.2441
Epoch 4, Avg Loss: 0.2342
Epoch 5, Avg Loss: 0.2214
Epoch 6, Avg Loss: 0.2084
Epoch 7, Avg Loss: 0.1965
Epoch 8, Avg Loss: 0.1846
Epoch 9, Avg Loss: 0.1811
Epoch 10, Avg Loss: 0.1712


In [50]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in test_loader:
        out = model(xb)
        probs = torch.softmax(out, dim=1)[:, 1]   # probability of FOG
        preds = (probs > 0.4).int()

        all_preds.extend(preds.numpy())
        all_labels.extend(yb.numpy())

In [51]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds))

[[1713  620]
 [  32 3628]]
              precision    recall  f1-score   support

           0       0.98      0.73      0.84      2333
           1       0.85      0.99      0.92      3660

    accuracy                           0.89      5993
   macro avg       0.92      0.86      0.88      5993
weighted avg       0.90      0.89      0.89      5993



In [52]:
print(y.mean())

0.5971967295177707
